In [1]:
import pandas as pd
import numpy as np

# 1. Load the dataset and parse timestamp columns properly
date_cols = [
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

df = pd.read_csv('olist_master_dataset.csv', parse_dates=date_cols)
print(f"Initial Row Count: {len(df):,}")

# 2. Deduplicate Orders (The "Multiple Reviews per Order" Fix)
# Sort so the lowest review score comes first (worst-case customer experience), then keep 1 row per order_id
df = df.sort_values(by=['order_id', 'review_score'], ascending=[True, True])
df = df.drop_duplicates(subset=['order_id'], keep='first').copy()
print(f"Deduplicated Row Count (100% unique orders): {len(df):,}")

# 3. Handle Missing Values (Nulls)
# If an order has no review score, we keep it as NaN for accurate averages, 
# but fill missing comment text with an empty string so text processing doesn't crash.
df['review_comment_message'] = df['review_comment_message'].fillna('').astype(str)

# Flag whether the customer actually wrote a text review (1 = Yes, 0 = No)
df['has_written_review'] = np.where(df['review_comment_message'].str.strip() != '', 1, 0)

# 4. Portuguese NLP Sentiment & Issue Categorization
# Define core Portuguese roots for positive, negative, and delivery-complaint terms
neg_words = ['atraso', 'atrasado', 'demora', 'demorou', 'péssimo', 'ruim', 'defeito', 'nunca', 'não recebi', 'incompleto']
pos_words = ['ótimo', 'excelente', 'bom', 'rápido', 'recomendo', 'perfeito', 'parabéns', 'adorei', 'antes do prazo']
delivery_words = ['entrega', 'prazo', 'correios', 'chegou', 'atraso', 'demora', 'frete', 'transportadora']

def classify_sentiment(row):
    text = row['review_comment_message'].lower().strip()
    score = row['review_score']
    
    # If they wrote a comment, check the text keywords first, then fall back to score
    if text != '':
        if any(w in text for w in neg_words) or score <= 2:
            return 'Negative'
        elif any(w in text for w in pos_words) or score >= 4:
            return 'Positive'
        else:
            return 'Neutral'
    # If they didn't write text, classify based on star rating
    else:
        if pd.isna(score):
            return 'No Review'
        elif score >= 4:
            return 'Positive'
        elif score == 3:
            return 'Neutral'
        else:
            return 'Negative'

def mentions_delivery_issue(row):
    text = row['review_comment_message'].lower().strip()
    # Flag if it's a negative review that specifically talks about shipping/delivery
    if row['sentiment'] == 'Negative' and any(w in text for w in delivery_words):
        return 1
    return 0

df['sentiment'] = df.apply(classify_sentiment, axis=1)
df['is_delivery_complaint'] = df.apply(mentions_delivery_issue, axis=1)

# 5. Quick Sanity Check Summary
print("\n--- Sentiment Breakdown ---")
print(df['sentiment'].value_counts())

print(f"\nTotal Negative Delivery Complaints Detected: {df['is_delivery_complaint'].sum():,}")

# Save the cleaned order-level table
df.to_csv('olist_orders_cleaned.csv', index=False)
print("\nSaved clean dataset to 'olist_orders_cleaned.csv'")

Initial Row Count: 96,999
Deduplicated Row Count (100% unique orders): 96,470

--- Sentiment Breakdown ---
sentiment
Positive     75525
Negative     13477
Neutral       6822
No Review      646
Name: count, dtype: int64

Total Negative Delivery Complaints Detected: 3,700

Saved clean dataset to 'olist_orders_cleaned.csv'


In [2]:
# 1. Set the "Snapshot Date" (1 day after the very last purchase in the dataset)
# Since Olist is historical data (ending in 2018), we can't use today's 2026 date, 
# otherwise every customer would look 8 years inactive!
snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f"Analysis Snapshot Date: {snapshot_date}")

# 2. Aggregate metrics at the Unique Customer level
rfm = df.groupby('customer_unique_id').agg(
    last_purchase_date=('order_purchase_timestamp', 'max'),
    frequency=('order_id', 'nunique'),
    monetary=('total_order_cost', 'sum'),
    avg_review_score=('review_score', 'mean'),
    customer_state=('customer_state', 'first')
).reset_index()

# Calculate Recency in whole days
rfm['recency_days'] = (snapshot_date - rfm['last_purchase_date']).dt.days
rfm['monetary'] = rfm['monetary'].round(2)
rfm['avg_review_score'] = rfm['avg_review_score'].round(2)

# 3. Assign R, F, and M Scores (Scale of 1 to 5, where 5 is best)
# Recency: Lower days is better, so labels are reversed [5, 4, 3, 2, 1]
rfm['R_Score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Monetary: Higher spend is better, so labels are [1, 2, 3, 4, 5]
rfm['M_Score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Frequency: Custom business rules (since ~97% of Olist buyers only purchase once)
def assign_f_score(freq):
    if freq == 1:
        return 1
    elif freq == 2:
        return 3
    else:
        return 5

rfm['F_Score'] = rfm['frequency'].apply(assign_f_score)

# Combine into an RFM string (e.g., "555") for reference
rfm['RFM_Code'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# 4. Map Scores to Actionable Marketing Segments
def segment_customer(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    
    if f >= 3 and r >= 4:
        return 'Champions (Repeat & Recent)'
    elif f >= 3 and r <= 3:
        return 'Loyal / Repeat (Needs Re-engagement)'
    elif r >= 4 and m >= 4:
        return 'High-Value New Customers'
    elif r >= 4 and m <= 3:
        return 'Recent First-Time Buyers'
    elif r == 3:
        return 'About to Sleep (Drifting)'
    elif r <= 2 and m >= 4:
        return 'At Risk (High Spend, Long Gone)'
    else:
        return 'Hibernating / Lost'

rfm['customer_segment'] = rfm.apply(segment_customer, axis=1)

# 5. Inspect the Results
print(f"\nTotal Unique Customers: {len(rfm):,}")
print("\n--- Customer Segment Breakdown ---")
segment_summary = rfm.groupby('customer_segment').agg(
    customer_count=('customer_unique_id', 'count'),
    avg_recency_days=('recency_days', 'mean'),
    avg_spend=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).round(2).sort_values(by='total_revenue', ascending=False)

print(segment_summary)

# Save Customer RFM Table for Power BI
rfm.to_csv('olist_customers_rfm.csv', index=False)
print("\nSaved customer segmentation table to 'olist_customers_rfm.csv'")

Analysis Snapshot Date: 2018-08-30 15:00:37

Total Unique Customers: 93,350

--- Customer Segment Breakdown ---
                                      customer_count  avg_recency_days  \
customer_segment                                                         
High-Value New Customers                       14452             91.61   
At Risk (High Spend, Long Gone)                13734            394.17   
About to Sleep (Drifting)                      18105            220.46   
Hibernating / Lost                             22492            396.07   
Recent First-Time Buyers                       21766             89.97   
Loyal / Repeat (Needs Re-engagement)            1592            319.88   
Champions (Repeat & Recent)                     1209             89.16   

                                      avg_spend  total_revenue  
customer_segment                                                
High-Value New Customers                 302.02     4364860.59  
At Risk (High Spend, Long 